# Lab 7 — Navigation with move_base

<svg width="100%" viewBox="0 0 1260 150" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Navigation stack">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L0,6 L9,3 z" fill="#334155"/></marker></defs>
<rect x="0" y="0" width="1260" height="150" rx="18" fill="#f8fafc" stroke="#cbd5e1"/>
<text x="24" y="30" font-family="Arial" font-size="20" font-weight="700" fill="#0f172a">Navigation stack</text>
<rect x="25.0" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="115.4" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Saved map</text>
<line x1="205.8" y1="84" x2="224.8" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="230.8" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="321.2" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Localization</text>
<line x1="411.7" y1="84" x2="430.7" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="436.7" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="527.1" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Global planner</text>
<line x1="617.5" y1="84" x2="636.5" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="642.5" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="732.9" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Local planner</text>
<line x1="823.3" y1="84" x2="842.3" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="848.3" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="938.8" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Velocity commands</text>
<line x1="1029.2" y1="84" x2="1048.2" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="1054.2" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="1144.6" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Robot motion</text>
</svg>


## Learning outcomes
You will load a saved map, initialize localization, send navigation goals in RViz, and write a waypoint script that publishes goal poses.

## Concept snapshot
The `move_base` navigation stack combines localization, costmaps, global planning, local planning, and recovery behaviors. The robot needs a map, an initial pose estimate, sensor updates, and safe controller limits before it can navigate reliably.


## Prepare the map
Copy a saved map pair into the map directory:
```bash
~/jetauto_ws/src/jetauto_slam/maps/
```
The pair should include:
- `room_map.pgm` or your map image
- `room_map.yaml` or matching metadata file


## Start the navigation stack
On the robot:
```bash
sudo systemctl stop start_app_node.service
roslaunch jetauto_controller jetauto_controller.launch
roslaunch jetauto_navigation navigation.launch map:=room_map
```
In another terminal:
```bash
roslaunch jetauto_navigation rviz_navigation.launch
```


## RViz workflow
1. Use **2D Pose Estimate** to align the robot with its real position on the map.
2. Use **2D Nav Goal** to send a goal pose.
3. Watch the global path and local costmap update.
4. If localization is poor, manually drive the robot slowly and reset the initial pose.
5. Test obstacle response only after the robot can navigate reliably without obstacles.


## Waypoint navigation starter script
Create `waypoint_navigation.py` in the `scripts/` folder of your own package. Confirm the goal topic with `rostopic list`; some setups use `/move_base_simple/goal`, while namespaced setups use `/jetauto_1/move_base_simple/goal`.
```python

#!/usr/bin/env python3
import math
import rospy
from geometry_msgs.msg import PoseStamped
from tf.transformations import quaternion_from_euler

WAYPOINTS = [
    (-0.8, 0.2, 0.0),
    (0.5, 1.0, 90.0),
    (1.4, 0.0, 180.0),
]


def make_goal(x, y, yaw_deg, frame_id='map'):
    goal = PoseStamped()
    goal.header.frame_id = frame_id
    goal.header.stamp = rospy.Time.now()
    goal.pose.position.x = x
    goal.pose.position.y = y
    qx, qy, qz, qw = quaternion_from_euler(0.0, 0.0, math.radians(yaw_deg))
    goal.pose.orientation.x = qx
    goal.pose.orientation.y = qy
    goal.pose.orientation.z = qz
    goal.pose.orientation.w = qw
    return goal


def main():
    rospy.init_node('waypoint_navigation')
    pub = rospy.Publisher('/jetauto_1/move_base_simple/goal', PoseStamped, queue_size=1)
    rospy.sleep(2.0)

    for i, (x, y, yaw) in enumerate(WAYPOINTS, start=1):
        input(f'Press Enter to send waypoint {i}: x={x}, y={y}, yaw={yaw} deg')
        pub.publish(make_goal(x, y, yaw))
        rospy.loginfo('Published waypoint %d', i)


if __name__ == '__main__':
    main()

```


## Lab tasks
- Navigate to at least one RViz goal manually.
- Run the waypoint script for at least two poses.
- Record what happens when localization starts with an incorrect pose.
- Explain how global and local planning behave differently.

## Troubleshooting
- No path appears: check map, initial pose, costmap, and goal frame.
- Robot spins or drifts: check localization and transforms.
- Robot refuses to move: verify controller, emergency stop, app service, and topic names.


---

## Lab Task Results

### Task 1 — Navigate to at least one RViz goal manually

**What we did:**
- Launched the navigation stack with `navigation.launch map:=explore` using the map from Project 4.
- Opened RViz with `rviz_navigation.launch`.
- Used **2D Pose Estimate** to set the robot's initial pose by clicking and dragging an arrow on the map at the robot's actual physical location.
- Used **2D Nav Goal** to send a target pose — clicked a point on the map and dragged to set the desired orientation.
- Observed the global planner compute a path (green line) and the local planner generate velocity commands to follow it.
- The robot successfully reached the goal, stopping within ~10 cm of the target.

    "Navigation videos are available on Google Drive:\n",


### Task 2 — Run the waypoint script for at least two poses

The script `waypoint_navigation.py` publishes `PoseStamped` messages to `/jetauto_1/move_base_simple/goal`. We tested it with two waypoints:

```
WAYPOINTS = [
    (-0.8, 0.2, 0.0),     # Waypoint 1: near origin, facing 0°
    (0.5, 1.0, 90.0),     # Waypoint 2: forward-right, facing 90°
]
```

**Observations:**
- After pressing Enter for each waypoint, `move_base` received the goal and the robot began navigating immediately.
- The global planner replanned the path when the robot deviated (e.g. due to local obstacle avoidance).
- The robot paused briefly between waypoints (goal reached → next goal published).
- Both waypoints were reached successfully.

### Task 3 — Record what happens when localization starts with an incorrect pose

**Experiment:** We deliberately set the initial pose via **2D Pose Estimate** to a location ~2 metre away from the robot's true position.

**Observed behaviour:**
- The laser scan (displayed in RViz) did **not align** with the walls on the map — the scan points were visibly offset from the black occupancy boundaries.
- The robot's estimated pose on the map drifted as it moved, because AMCL was trying to match scan data against the wrong region of the map.
- `move_base` computed a path based on the **incorrect** pose, so the global plan did not correspond to the robot's actual environment — the path passed through walls on the map.
- The local planner's costmap also reflected the misalignment: obstacles that were detected by the LiDAR appeared in the wrong location relative to the map.
- **Conclusion:** If the initial pose is wrong, the entire navigation pipeline fails. AMCL will eventually converge if the robot moves slowly through a feature‑rich area (giving it enough distinct scan features to re‑localise), but navigation should not be attempted until the laser scan visibly aligns with the map walls in RViz.

### Task 4 — Explain how global and local planning behave differently

| Aspect | Global Planner | Local Planner |
|---|---|---|
| **Algorithm** | Dijkstra / A\* (in our case: `global_planner/GlobalPlanner`) | Dynamic Window Approach (DWA) — `dwa_local_planner/DWAPlannerROS` |
| **Input** | Static costmap (obstacles from the saved map + any new sensor data marked as static) | Local costmap (a rolling window around the robot, updated in real time with laser scans) |
| **Output** | A long‑range path from robot to goal — a sequence of `(x, y, θ)` waypoints that avoids known static obstacles | A short‑horizon velocity command `(v_x, v_y, ω_z)` that follows the global path while avoiding dynamic obstacles |
| **Replanning** | Only when the global costmap changes significantly or the goal is updated (relatively infrequent) | Continuously — at ~5–20 Hz — reacting to new obstacles, people, or path deviations |
| **Scope** | Whole map — knows about walls, fixed obstacles, and the full environment | Local window (~3–5 m radius) — only cares about what is immediately around the robot |
| **Failure mode** | Cannot find a path if the goal is unreachable (e.g. inside a wall or blocked by static obstacles) | Cannot find a safe velocity if an obstacle is too close — robot stops and may trigger recovery behaviours (rotate in place, back up) |

**Key insight:** The global planner provides the *strategic* route ("go through this corridor, turn left at the intersection"), while the local planner handles *tactical* execution ("swerve around this chair, slow down near the wall"). Neither can work without the other — a global plan without local execution would collide with dynamic obstacles; a local planner without a global plan would get stuck in local minima.

## Waypoint Navigation Script (complete)

Save as `waypoint_navigation.py` in your package's `scripts/` folder. Make executable with `chmod +x waypoint_navigation.py`.

In [ ]:
#!/usr/bin/env python3
"""Publish a sequence of navigation waypoints to move_base."""
import math
import rospy
from geometry_msgs.msg import PoseStamped
from tf.transformations import quaternion_from_euler

# ── Define your waypoints: (x, y, yaw_degrees) ──
WAYPOINTS = [
    (-0.8, 0.2, 0.0),      # Pose 1: near origin, facing east
    (0.5, 1.0, 90.0),      # Pose 2: forward-right, facing north
    (1.4, 0.0, 180.0),     # Pose 3: far end, facing west
]

GOAL_TOPIC = '/jetauto_1/move_base_simple/goal'


def make_goal(x, y, yaw_deg, frame_id='map'):
    """Build a PoseStamped goal from (x, y, yaw_degrees)."""
    goal = PoseStamped()
    goal.header.frame_id = frame_id
    goal.header.stamp = rospy.Time.now()
    goal.pose.position.x = x
    goal.pose.position.y = y
    goal.pose.position.z = 0.0
    q = quaternion_from_euler(0.0, 0.0, math.radians(yaw_deg))
    goal.pose.orientation.x = q[0]
    goal.pose.orientation.y = q[1]
    goal.pose.orientation.z = q[2]
    goal.pose.orientation.w = q[3]
    return goal


def main():
    rospy.init_node('waypoint_navigation')
    pub = rospy.Publisher(GOAL_TOPIC, PoseStamped, queue_size=1)
    # Wait for publisher to be ready and for move_base to initialise
    rospy.sleep(2.0)

    for i, (x, y, yaw) in enumerate(WAYPOINTS, start=1):
        raw_input(f'Press Enter to send waypoint {i}: x={x}, y={y}, yaw={yaw} deg '
                   f'(Ctrl-C to quit)'  if hasattr(__builtins__, 'raw_input')
                   else input(f'Press Enter to send waypoint {i}: x={x}, y={y}, yaw={yaw} deg '))
        goal = make_goal(x, y, yaw)
        pub.publish(goal)
        rospy.loginfo(f'Published waypoint {i}: ({x:.1f}, {y:.1f}, {yaw:.0f}°)')


if __name__ == '__main__':
    main()

---

## Troubleshooting Log

| Symptom | Likely Cause | Resolution |
|---|---|---|
| No path appears in RViz | Map not loaded or goal in unknown/inflated cell | Check `rostopic echo /move_base/global_costmap/cost_map` ; ensure goal is in free (white) space, not inside a wall |
| Robot spins in place | AMCL has not converged — particle cloud is too spread | Re‑set **2D Pose Estimate** more accurately; drive robot slowly forward ~0.5 m to help AMCL converge |
| Robot drives into wall | Incorrect initial pose — laser scan does not match map | Check laser scan overlay in RViz; re‑initialise pose until scan aligns with walls |
| Robot refuses to move | `start_app_node.service` still running; emergency stop; or controller not launched | Run `sudo systemctl stop start_app_node.service` ; verify `roslaunch jetauto_controller jetauto_controller.launch` is active |
| Path oscillates / robot wiggles | Local planner parameters too aggressive | Reduce `max_vel_x`, `acc_lim_x`, or increase `path_distance_bias` in DWA params |
| Goal reached too early | `xy_goal_tolerance` too large | Lower `xy_goal_tolerance` to ~0.1 m in `DWAPlannerROS` params |

## Navigation Evidence

Navigation videos are available on Google Drive:

* [IMG_4183.MOV — robot navigating to goal](https://drive.google.com/file/d/1MW1dcmuNh606nZTBmq_qdjX6pyi9EXUP/view)
* [IMG_4190.MOV — waypoint sequence](https://drive.google.com/file/d/1qxXMp6CwqVQMJx4-BTOeLMSD8ANImquV/view)
* [IMG_4192.MOV — navigation with obstacles](https://drive.google.com/file/d/12_9L4q9O4UTGFaGaMorH240ZkcAi2Mj5/view)

The map used for navigation is the `explore.pgm` / `explore.yaml` pair generated in Project 4.